# 내 목소리와 잡음 살펴보기
*Noise Reduction Challenge · Dataset Exploration*

수강생이 녹음한 음성에 다양한 환경 잡음을 더한 **Clean–Noisy 쌍**입니다.
Train으로 알고리즘을 개발하고, Validation으로 파라미터와 성능을 비교하세요.
잡음 종류와 SNR을 선택해 소리를 듣고, 시간·주파수별 차이를 살펴봅니다.

| 배포 데이터 | Train | Validation | 전체 |
| :--- | ---: | ---: | ---: |
| 음성 쌍 수 | 1,296 | 164 | 1,460 |
| WAV 파일 수 (Clean + Noisy) | 2,592 | 328 | 2,920 |
| 화자 수 | 35 | 5 | 40 |
| 총 음성 길이 (쌍당 한 번 계산) | 64.8분 | 8.2분 | 73.0분 |

- **오디오:** 16,000 Hz (16 kHz), 모노, WAV / 16-bit PCM.
- **파일 길이:** 모두 3초, 파일당 48,000개 샘플. Clean과 Noisy는 길이와 시간 위치가 대응합니다.
- **잡음:** 23개 종류, 난이도 `easy / moderate / hard`, 목표 SNR `0 / 5 / 10 / 15 dB`.
- **분할:** Train과 Validation의 화자는 겹치지 않습니다. Clean은 비교·평가의 정답이며, 잡음 제거 알고리즘의 입력은 Noisy입니다.

위 수치는 현재 배포본 기준입니다. **2절**에서 경로를 설정하면 실제 `samples.csv`를 읽어 요약 표를 다시 계산합니다.
**5절**에서는 드롭다운과 버튼으로 한 샘플씩 시각화·청취합니다.


## 1. 실행 환경

Colab 또는 로컬 Jupyter에서 실행합니다. 아래 셀은 현재 환경에 없는 패키지만 설치합니다.


In [ ]:
import os
import sys
from pathlib import Path
import importlib.util
import subprocess

# Make Conda DLLs available when the notebook kernel is started without activation.
_conda_dll_handles = []
if sys.platform == "win32" and (Path(sys.prefix) / "conda-meta").is_dir():
    _conda_bin = Path(sys.prefix) / "Library" / "bin"
    if _conda_bin.is_dir():
        os.environ["PATH"] = str(_conda_bin) + os.pathsep + os.environ.get("PATH", "")
        _conda_dll_handles.append(os.add_dll_directory(str(_conda_bin)))

dependencies = {
    "numpy": "numpy>=1.26,<3", "scipy": "scipy>=1.12,<2",
    "soundfile": "soundfile>=0.12,<1", "pandas": "pandas>=2.1,<4",
    "matplotlib": "matplotlib>=3.8,<4", "IPython": "ipython>=8,<10",
    "ipywidgets": "ipywidgets>=8,<9",
}
missing = [package for module, package in dependencies.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal
import matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display, clear_output

print("Python:", sys.version.split()[0])


## 2. 데이터 경로

- **로컬:** 기본값은 `release/noise_reduction_challenge`이며, 압축을 푼 폴더 안에서 실행해도 자동으로 찾습니다.
- **Colab:** 폴더 전체를 `내 드라이브/noise_reduction_challenge`에 올리고 `DATA_SOURCE="google_drive"`로 설정하세요.

각 split의 `samples.csv`는 샘플 선택에 필요한 목록입니다. 데이터 폴더와 함께 보관하세요.


In [ ]:
DATA_SOURCE = "local"  # "local" 또는 "google_drive"
LOCAL_DATA_PATH = Path("release/noise_reduction_challenge")
GOOGLE_DRIVE_DATA_PATH = Path("/content/drive/MyDrive/noise_reduction_challenge")
N_FFT = 512  # 16 kHz에서 32 ms
HOP = 128    # 프레임 간격 8 ms (75% overlap)


In [ ]:
def resolve_local_data_path(data_path="release/noise_reduction_challenge", start=None):
    """Resolve an explicit local path from the working folder or its parents."""
    data_path = Path(data_path).expanduser()
    start = Path(start or Path.cwd()).resolve()
    candidates = ([data_path] if data_path.is_absolute() else
                  [parent / data_path for parent in (start, *start.parents)])
    # The default also works after extracting the folder outside this workspace.
    if data_path == Path("release/noise_reduction_challenge"):
        for parent in (start, *start.parents):
            candidates.extend((parent, parent / "noise_reduction_challenge"))
    for candidate in candidates:
        if all((candidate / split / "samples.csv").is_file()
               for split in ("train", "validation")):
            return candidate.resolve()
    raise FileNotFoundError(
        f"LOCAL_DATA_PATH={str(data_path)!r} could not be found from {start}. "
        "Set it to the folder containing train and validation, each with samples.csv."
    )

def load_manifest(root, split):
    if split not in ("train", "validation"):
        raise ValueError("split must be train or validation")
    root = Path(root).resolve()
    df = pd.read_csv(root / split / "samples.csv")
    required = {"segment_id", "clean_path", "noisy_path", "sample_rate_hz", "num_samples",
                "noise_category", "noise_collection", "noise_code", "noise_difficulty",
                "target_snr_db", "split"}
    if not required.issubset(df.columns) or df.empty:
        raise ValueError("Empty manifest or required columns missing.")
    if df.segment_id.duplicated().any():
        raise ValueError("Duplicate segment IDs.")
    if "split" in df and not df["split"].eq(split).all():
        raise ValueError("Manifest split mismatch.")
    for column in ("clean_path", "noisy_path"):
        for relative in df[column]:
            path = (root / relative).resolve()
            if not path.is_relative_to(root) or not path.is_file():
                raise FileNotFoundError(f"Invalid/missing file: {path}")
    return df

def select_examples(manifest, noise_category=None, noise_code=None,
                    target_snr_db=None, noise_difficulty=None):
    """Select metadata rows only. None means all values; no waveform is processed."""
    selected = manifest.copy()
    for column, value in (("noise_category", noise_category), ("noise_code", noise_code),
                           ("noise_difficulty", noise_difficulty)):
        if value is not None:
            selected = selected[selected[column].eq(value)]
    if target_snr_db is not None:
        selected = selected[np.isclose(selected.target_snr_db, float(target_snr_db))]
    return selected.sort_values("segment_id").reset_index(drop=True)


In [ ]:
if DATA_SOURCE == "local":
    DATA_ROOT = resolve_local_data_path(LOCAL_DATA_PATH)
elif DATA_SOURCE == "google_drive":
    try:
        from google.colab import drive
    except ImportError as error:
        raise RuntimeError("google_drive는 Colab용입니다. 로컬에서는 local과 폴더 경로를 사용하세요.") from error
    drive.mount("/content/drive")
    DATA_ROOT = GOOGLE_DRIVE_DATA_PATH.expanduser().resolve()
else:
    raise ValueError('DATA_SOURCE는 "local" 또는 "google_drive"여야 합니다.')

DATASETS = {split: load_manifest(DATA_ROOT, split) for split in ("train", "validation")}
print("DATA_ROOT:", DATA_ROOT)
print("Train:", len(DATASETS["train"]), "| Validation:", len(DATASETS["validation"]))


### 데이터 정보 확인

`samples.csv`의 `segment_id`는 음성 쌍 ID, `clean_path`·`noisy_path`는 데이터 루트 기준 상대 경로입니다.
`sample_rate_hz`는 초당 샘플 수, `num_samples / sample_rate_hz`는 파일 길이(초)입니다.
`noise_category`·`noise_code`·`noise_collection`은 잡음의 종류·코드·출처를,
`noise_difficulty`는 난이도 분류, `target_snr_db`는 혼합 시 설정한 목표 SNR을 나타냅니다.
SNR이 낮을수록 음성에 비해 잡음이 강합니다. 저장된 WAV로 계산한 SNR은 목표값과 조금 다를 수 있습니다.

아래 표는 목록에서 계산합니다. 총 길이는 Clean과 Noisy를 중복 합산하지 않습니다.


In [ ]:
def summarize_dataset(datasets):
    rows = []
    groups = list(datasets.items()) + [
        ("all", pd.concat(datasets.values(), ignore_index=True))
    ]
    for split, table in groups:
        duration = table["num_samples"] / table["sample_rate_hz"]
        speakers = table["segment_id"].str.extract(r"^(speaker\d+)_", expand=False)
        rows.append({
            "Split": split,
            "음성 쌍": len(table),
            "WAV 파일": 2 * len(table),
            "화자": speakers.nunique(),
            "Sampling rate (Hz)": sorted(table["sample_rate_hz"].unique().tolist()),
            "길이 최소/평균/최대 (s)": [round(v, 3) for v in
                                         (duration.min(), duration.mean(), duration.max())],
            "총 길이 (min)": round(duration.sum() / 60, 2),
            "잡음 종류": table["noise_category"].nunique(),
            "목표 SNR (dB)": sorted(table["target_snr_db"].unique().tolist()),
        })
    return pd.DataFrame(rows).set_index("Split")

dataset_summary = summarize_dataset(DATASETS)
display(dataset_summary)


### Clean + Noise → Noisy

Validation의 `speaker015_utt01_seg0003` (alarm_siren / N035, 목표 SNR 5 dB)에서 가져온 고정 예시입니다.
세 스펙트로그램의 색상 범위와 dB 기준을 동일하게 맞췄습니다.


In [ ]:
from IPython.display import Image
mixture_figure = DATA_ROOT / "figures" / "speech_noise_mixture.png"
if not mixture_figure.is_file():
    raise FileNotFoundError(f"그림 파일을 찾을 수 없습니다: {mixture_figure}. figures 폴더도 함께 복사하세요.")
display(Image(filename=str(mixture_figure), width=1200))


$$\underbrace{y[n]}_{\text{Noisy}} = \underbrace{s[n]}_{\text{Clean}} + \underbrace{d[n]}_{\text{Added noise}}$$
$$Y(k,m)=S(k,m)+D(k,m)$$

$n$은 시간 샘플, $k$는 주파수 bin, $m$은 STFT 프레임입니다.
$d[n]$은 이미 혼합 레벨이 반영된 잡음이며, 그림에서는 배포 파일의 `Noisy − Clean`으로 구했습니다.
**덧셈은 파형과 복소 STFT에서 성립합니다.** 그림의 dB 값이나 파워 $|S|^2$, $|D|^2$를 그대로 더하면 Noisy 그림이 되는 것은 아닙니다.


## 3. 어떤 잡음이 들어 있을까요?

표와 그래프에서 잡음 종류별 샘플 수를 확인하세요. **SNR이 낮을수록 음성에 비해 잡음이 강합니다.**
샘플 수는 음성 파일 쌍의 개수입니다.


In [ ]:
all_samples = pd.concat(DATASETS.values(), ignore_index=True)
noise_counts = pd.crosstab(all_samples.noise_category, all_samples["split"]).reindex(
    columns=["train", "validation"], fill_value=0
).sort_index()
noise_counts.index.name = "noise_category"
display(noise_counts)
display(all_samples.groupby(["split", "noise_category"]).agg(
    samples=("segment_id", "size"),
    noise_codes=("noise_code", lambda x: ", ".join(sorted(x.unique()))),
    snr_choices_db=("target_snr_db", lambda x: sorted(x.unique().tolist())),
).reset_index())

fig, axes = plt.subplots(1, 2, figsize=(13, 7), sharey=True, constrained_layout=True)
for ax, split, color in zip(axes, ("train", "validation"), ("#426FA6", "#159DAD")):
    ax.barh(noise_counts.index, noise_counts[split], color=color)
    ax.set(title=split.capitalize(), xlabel="Number of samples")
    ax.grid(axis="x", alpha=0.2)
axes[0].invert_yaxis()
display(fig)
plt.close(fig)


## 4. 청취·스펙트로그램 함수

**STFT**는 겹치는 짧은 구간마다 창 함수를 적용하고 FFT를 계산하여 시간에 따른 주파수 변화를 보여줍니다.
아래 세 그림은 **같은 색상·dB 기준**, 재생 버튼은 **같은 gain**을 사용합니다.

`Added noise = Noisy − Clean`은 배포된 파일에 섞인 잡음을 확인하기 위한 근사치입니다.
원본 잡음 파일과는 레벨이 다르며 WAV 양자화 오차도 포함될 수 있습니다.


In [ ]:
def inspect_sample(row, data_root=DATA_ROOT, n_fft=512, hop=128):
    """Read one aligned pair and display three spectrograms/audio players."""
    if not isinstance(n_fft, int) or n_fft < 4 or not isinstance(hop, int) or not 0 < hop <= n_fft:
        raise ValueError("n_fft >= 4, 0 < hop <= n_fft인 정수를 사용하세요.")
    root = Path(data_root).resolve()
    audio = {}
    sample_rate = int(row["sample_rate_hz"])
    for kind in ("noisy", "clean"):
        path = (root / row[f"{kind}_path"]).resolve()
        if not path.is_relative_to(root):
            raise ValueError("데이터 폴더 밖의 경로입니다.")
        values, sr = sf.read(path, dtype="float64")
        if sr != sample_rate or values.ndim != 1:
            raise ValueError(f"표본률 또는 채널 수가 다릅니다: {path}")
        if len(values) != int(row["num_samples"]) or not np.isfinite(values).all():
            raise ValueError(f"길이 또는 음성 값이 잘못되었습니다: {path}")
        audio[kind] = values
    if audio["noisy"].shape != audio["clean"].shape or len(audio["noisy"]) < n_fft:
        raise ValueError("Clean/Noisy 길이가 같고 n_fft 이상이어야 합니다.")
    audio["added_noise"] = audio["noisy"] - audio["clean"]

    spectra = {}
    for name, values in audio.items():
        frequencies, times, z = signal.stft(
            values, fs=sample_rate, window="hann", nperseg=n_fft,
            noverlap=n_fft-hop, nfft=n_fft, boundary="zeros", padded=True,
        )
        spectra[name] = np.abs(z)
    reference = max(max(float(z.max()) for z in spectra.values()), 1e-12)
    labels = {"noisy": "Noisy speech", "clean": "Clean speech",
              "added_noise": "Added noise (Noisy - Clean)"}
    fig, axes = plt.subplots(3, 1, figsize=(12, 7.5), sharex=True, sharey=True,
                             constrained_layout=True)
    for ax, (name, magnitude) in zip(axes, spectra.items()):
        db = 20 * np.log10(np.maximum(magnitude / reference, 1e-6))
        mesh = ax.pcolormesh(times, frequencies/1000, db, shading="auto",
                             cmap="magma", vmin=-80, vmax=0)
        ax.set(title=labels[name], ylabel="Frequency (kHz)",
               xlim=(0, len(audio["noisy"])/sample_rate), ylim=(0, sample_rate/2000))
    axes[-1].set_xlabel("Time (s)")
    fig.colorbar(mesh, ax=list(axes), label="dB relative to shared STFT peak")
    try:
        display(Markdown(f"**{row['segment_id']}** · {row['noise_category']} / "
                         f"{row['noise_code']} · target SNR {row['target_snr_db']:g} dB"))
        display(fig)
    finally:
        plt.close(fig)

    # One shared gain preserves the relative levels of speech and added noise.
    gain = 0.99 / max(1.0, max(float(np.abs(x).max()) for x in audio.values()))
    for name, values in audio.items():
        display(Markdown(f"**{labels[name]}**"))
        display(Audio(values * gain, rate=sample_rate, normalize=False))
    return {"audio": audio, "sample_rate": sample_rate, "playback_gain": gain}


## 5. 잡음별 샘플 탐색

Split·Noise·Code·SNR·Difficulty·Sample을 선택하고 **듣기 / 보기**를 누르세요.
선택할 때는 목록만 갱신하고, 버튼을 누를 때 한 쌍만 읽어 Noisy·Clean·혼합 잡음을 표시합니다.
잡음이 특정 주파수에 집중되는지, 넓은 대역에 퍼지는지, 시간에 따라 달라지는지 비교해 보세요.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output


def create_sample_browser(datasets, on_sample, on_path=None, button_label="듣기 / 보기"):
    """Filter index rows only; read audio/infer only after a button click."""
    all_rows = pd.concat(datasets.values(), ignore_index=True)
    split = widgets.Dropdown(options=list(datasets), value="validation", description="Split:")
    categories = sorted(all_rows.noise_category.unique())
    noise = widgets.Dropdown(options=[("전체", None)] + [(x, x) for x in categories],
        value="alarm_siren" if "alarm_siren" in categories else None, description="Noise:")
    code = widgets.Dropdown(options=[("전체", None)] + [(x, x) for x in sorted(all_rows.noise_code.unique())],
                            description="Code:")
    snr = widgets.Dropdown(options=[("전체", None)] + [(f"{x:g} dB", float(x))
        for x in sorted(all_rows.target_snr_db.unique())], description="SNR:")
    difficulty = widgets.Dropdown(options=[("전체", None)] + [(x, x)
        for x in sorted(all_rows.noise_difficulty.unique())], description="Difficulty:")
    sample = widgets.Dropdown(description="Sample:", layout=widgets.Layout(width="95%"))
    status = widgets.Label()
    run = widgets.Button(description=button_label, button_style="primary",
                         layout=widgets.Layout(width="220px"))
    output = widgets.Output()
    filters = widgets.VBox([widgets.HBox([split, noise]), widgets.HBox([code, snr, difficulty]), sample])
    mode = widgets.ToggleButtons(options=["데이터셋", "파일 경로"], description="선택 방식:") if on_path else None
    noisy_path = widgets.Text(description="Noisy:", placeholder="noisy 파일 경로",
                              layout=widgets.Layout(width="95%"))
    clean_path = widgets.Text(description="Clean:", placeholder="선택 사항: 대응하는 clean 파일 경로",
                              layout=widgets.Layout(width="95%"))
    paths = widgets.VBox([noisy_path, clean_path]) if on_path else None
    candidates = pd.DataFrame()
    busy = False

    def sync_controls(change=None):
        path_mode = on_path is not None and mode.value == "파일 경로"
        filters.layout.display = "none" if path_mode else ""
        if paths is not None:
            paths.layout.display = "" if path_mode else "none"
        ready = bool(noisy_path.value.strip()) if path_mode else not candidates.empty
        run.disabled = busy or not ready
        status.value = ("상대 경로는 DATA_ROOT 기준입니다. Noisy 경로를 입력하세요." if path_mode
                        else f"조건에 맞는 샘플 {len(candidates)}개 — 선택 후 버튼을 누르세요.")
        output.clear_output()

    def refresh_choices(change=None):
        nonlocal candidates
        candidates = select_examples(datasets[split.value], noise_category=noise.value,
            noise_code=code.value, target_snr_db=snr.value, noise_difficulty=difficulty.value)
        sample.options = [(f"{i}: {row.segment_id} | {row.noise_code} | {row.target_snr_db:g} dB", i)
                          for i, row in candidates.iterrows()]
        sample.value = 0 if len(candidates) else None
        sync_controls()

    def preview_selected(button=None):
        nonlocal busy
        if run.disabled:
            return
        busy = True
        run.disabled = True
        try:
            with output:
                clear_output(wait=True)
                try:
                    if on_path is not None and mode.value == "파일 경로":
                        on_path(noisy_path.value.strip(), clean_path.value.strip() or None)
                    elif sample.value is not None:
                        on_sample(candidates.iloc[sample.value])
                except (ValueError, FileNotFoundError) as error:
                    print(f"입력을 확인하세요: {error}")
        finally:
            busy = False
            path_mode = on_path is not None and mode.value == "파일 경로"
            run.disabled = not (bool(noisy_path.value.strip()) if path_mode else not candidates.empty)

    for control in (split, noise, code, snr, difficulty):
        control.observe(refresh_choices, names="value")
    for control in ([sample, mode, noisy_path, clean_path] if on_path else [sample]):
        control.observe(sync_controls, names="value")
    run.on_click(preview_selected)
    refresh_choices()
    controls = ([mode, filters, paths] if on_path is not None else [filters])
    return widgets.VBox([*controls, status, run, output])


sample_browser = create_sample_browser(
    DATASETS,
    on_sample=lambda row: inspect_sample(row, data_root=DATA_ROOT, n_fft=N_FFT, hop=HOP),
)
display(sample_browser)


위젯을 사용할 수 없는 환경에서는 아래 코드를 새 셀에서 실행하세요.

```python
row = select_examples(DATASETS["validation"], noise_category="wind").iloc[0]
sample_result = inspect_sample(row, data_root=DATA_ROOT, n_fft=N_FFT, hop=HOP)
```
